# GlowLab · 01 — Dataset: tayyor datasetlarni birlashtirish

Bir nechta ochiq datasetdan **faqat bizga kerakli sinflarni** olamiz: `cleanser`, `toner`, `serum`, `moisturizer`, `sunscreen`, `lip`.

Manba datasetlar **object detection** (rasmda mahsulot atrofida ramka bor). Biz har bir ramkani **kesib olib (crop)** alohida rasm qilamiz → classification uchun tayyor.

**Qadamlar:** Drive → manbalarni tekshirish → crop → (ixtiyoriy) Bing bilan to'ldirish → qo'lda ko'rish → train/val/test.

Datasetlarni yuklash yo'riqnomasi: `ml/README.md`. Litsenziya: CC BY 4.0 (manbalar README'da ko'rsatilgan).

In [ ]:
from google.colab import drive   # Colab'ga Google Drive'ni ulash
drive.mount('/content/drive')

ROOT = '/content/drive/MyDrive/glowlab_ml'
SOURCES = f'{ROOT}/sources'   # Siz yuklagan datasetlar (har biri alohida papkada)
RAW = f'{ROOT}/raw'           # Kesib olingan rasmlar: raw/<sinf>/
DATA = f'{ROOT}/data'         # Yakuniy bo'lingan dataset: data/train|val|test/<sinf>/

In [ ]:
import os, glob, json, re, random, shutil, hashlib
from collections import Counter, defaultdict
from PIL import Image, ImageOps
import matplotlib.pyplot as plt

CLASSES = ['cleanser', 'toner', 'serum', 'moisturizer', 'sunscreen', 'lip']  # Ilovadagi kategoriyalar bilan bir xil nomlar (tartib MUHIM!)

def norm(name):
    """Sinf nomini solishtirish uchun soddalashtiramiz: 'Lip Stick' / 'lip-stick' → 'lipstick'"""
    return re.sub(r'[^a-z]', '', name.lower())

# Manba sinf nomi (norm qilingan) → bizning sinf. Ro'yxatda yo'qlari e'tiborga olinmaydi.
CLASS_MAP = {
    'cleanser': 'cleanser',
    'toner': 'toner', 'brighteningtoner': 'toner', 'soothingtoner': 'toner',
    'serum': 'serum',
    'moisturizer': 'moisturizer', 'lotion': 'moisturizer',
    'sunscreen': 'sunscreen', 'sunblock': 'sunscreen',
    'lip': 'lip', 'lipstick': 'lip', 'lipgloss': 'lip', 'lipbalm': 'lip',
}

## 1. Manbalarni tekshirish
Har bir manbada qaysi sinflar va nechta ramka borligini ko'rsatadi. O'ng ustun — bizning qaysi sinfga tushishi (`—` = tashlanadi).
Agar kerakli sinf `—` bo'lib qolsa, uni `CLASS_MAP` ga qo'shing.

In [ ]:
def find_coco(src):
    # Roboflow COCO eksporti: train/ valid/ test/ ichida _annotations.coco.json
    return sorted(glob.glob(f'{SOURCES}/{src}/**/_annotations.coco.json', recursive=True))

sources = sorted(d for d in os.listdir(SOURCES) if os.path.isdir(f'{SOURCES}/{d}'))
assert sources, "sources/ bo'sh — README'dagi yo'riqnoma bo'yicha datasetlarni yuklang"

for src in sources:
    files = find_coco(src)
    counts = Counter()
    for jf in files:
        coco = json.load(open(jf))
        names = {c['id']: c['name'] for c in coco['categories']}
        counts.update(names[a['category_id']] for a in coco['annotations'])
    print(f'\n== {src}: {len(files)} ta annotation fayl')
    if not files:
        print('  ⚠️ _annotations.coco.json topilmadi — COCO formatda yuklanganini tekshiring')
    for name, n in counts.most_common():
        print(f'  {name:28s} {n:5d} → {CLASS_MAP.get(norm(name), "—")}')

## 2. Crop: ramkalarni kesib olish
- ramka atrofiga 10% joy qo'shiladi (mahsulot chetlari kesilib qolmasin)
- 64px dan kichik crop'lar tashlanadi
- Roboflow bitta rasmning **augment qilingan nusxalarini** ham qo'shadi (`nom_jpg.rf.xxxx.jpg`) → har asl rasmdan faqat bittasini olamiz. Aks holda bir xil rasm train'da ham, test'da ham bo'lib, natija soxta yuqori chiqadi
- deyarli bir xil crop'lar (hash) → bittasi qoladi

In [ ]:
PAD = 0.10      # Ramka atrofiga qo'shiladigan joy
MIN_CROP = 64   # px — bundan kichik crop o'qitishga yaramaydi

if os.path.exists(RAW):
    shutil.rmtree(RAW)            # Qayta ishga tushirilganda toza boshlaymiz
for c in CLASSES:
    os.makedirs(f'{RAW}/{c}', exist_ok=True)

seen_hash = set()                 # Crop'larning "barmoq izlari" (takroriylarni ushlash uchun)
stats, skipped = Counter(), Counter()

for src in sources:
    seen_base = set()             # Shu manbada ko'rilgan asl rasm nomlari
    for jf in find_coco(src):
        folder = os.path.dirname(jf)
        coco = json.load(open(jf))
        names = {c['id']: c['name'] for c in coco['categories']}
        by_img = defaultdict(list)
        for a in coco['annotations']:
            by_img[a['image_id']].append(a)

        for info in coco['images']:
            base = info['file_name'].split('.rf.')[0]       # Augment nusxalari bir xil boshlanadi
            if base in seen_base:
                skipped['augment_copy'] += 1
                continue
            seen_base.add(base)
            anns = [a for a in by_img[info['id']] if norm(names[a['category_id']]) in CLASS_MAP]
            if not anns:
                continue                                    # Bu rasmda bizga kerakli mahsulot yo'q
            try:
                img = ImageOps.exif_transpose(Image.open(os.path.join(folder, info['file_name']))).convert('RGB')
            except Exception:
                skipped['broken'] += 1
                continue
            jw, jh = info['width'], info['height']              # Ramkalar shu o'lcham uchun belgilangan
            if img.size == (jh, jw) and jw != jh:
                img = img.transpose(Image.Transpose.ROTATE_270)  # Fayl 90° burilgan saqlangan → soat yo'nalishida 90° qaytaramiz
                skipped['rotated_fixed'] += 1                    # (xato emas — nechta rasm burilganini ko'rish uchun)
            W, H = img.size
            sx, sy = W / jw, H / jh                              # O'lcham boshqacha bo'lsa ramkani moslashtiramiz
            group = src[:12] + '__' + re.sub(r'[^A-Za-z0-9]+', '-', base)[:60]   # Asl rasm "guruhi"
            for a in anns:
                x, y, w, h = a['bbox']                      # COCO: chap-yuqori x, y + kenglik, balandlik
                x, w, y, h = x * sx, w * sx, y * sy, h * sy
                box = (max(0, int(x - w * PAD)), max(0, int(y - h * PAD)),
                       min(W, int(x + w + w * PAD)), min(H, int(y + h + h * PAD)))
                if min(box[2] - box[0], box[3] - box[1]) < MIN_CROP:
                    skipped['too_small'] += 1
                    continue
                crop = img.crop(box)
                digest = hashlib.md5(crop.resize((64, 64)).tobytes()).hexdigest()
                if digest in seen_hash:
                    skipped['duplicate'] += 1
                    continue
                seen_hash.add(digest)
                cls = CLASS_MAP[norm(names[a['category_id']])]
                crop.save(f"{RAW}/{cls}/{group}__{a['id']}.jpg", 'JPEG', quality=92)
                stats[cls] += 1

print('Kesib olindi:')
for c in CLASSES:
    print(f'  {c:12s} {stats[c]:5d}')
print("O'tkazib yuborildi:", dict(skipped))

## 3. (Ixtiyoriy) Kam sinflarni Bing rasmlari bilan to'ldirish
Biror sinfda `TARGET` dan kam rasm bo'lsa, yetishmaganini Bing'dan yuklaydi. Kerak bo'lsa `USE_BING = True` qiling.
⚠️ Internet rasmlari faqat o'quv maqsadida — datasetni tarqatmang.

In [ ]:
USE_BING = False   # True → kam sinflarni to'ldirish
TARGET = 150       # Har sinfda kamida shuncha rasm bo'lishi maqsad

QUERIES = {
    'cleanser':    ['facial cleanser bottle', 'foam cleanser tube'],
    'toner':       ['facial toner bottle', 'korean toner bottle'],
    'serum':       ['face serum dropper bottle', 'ampoule serum product'],
    'moisturizer': ['face moisturizer jar', 'face cream jar product'],
    'sunscreen':   ['sunscreen tube spf 50', 'korean sunscreen product'],
    'lip':         ['lip balm product', 'lipstick product photo'],
}

if USE_BING:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'bing-image-downloader'], check=True)
    from bing_image_downloader import downloader

    for cls in CLASSES:
        need = TARGET - len(os.listdir(f'{RAW}/{cls}'))
        if need <= 0:
            continue                                        # Bu sinf yetarli
        per_query = need // len(QUERIES[cls]) + 15          # Tozalashda yo'qoladiganlar uchun zaxira
        added = 0
        for q in QUERIES[cls]:
            downloader.download(q, limit=per_query, output_dir='/content/tmp_dl',
                                adult_filter_off=False, force_replace=False, timeout=30)
            folder = f'/content/tmp_dl/{q}'
            for i, name in enumerate(sorted(os.listdir(folder))):
                try:
                    with Image.open(f'{folder}/{name}') as im:
                        im = im.convert('RGB')
                        if min(im.size) < 150:
                            continue
                        digest = hashlib.md5(im.resize((64, 64)).tobytes()).hexdigest()
                        if digest in seen_hash:
                            continue
                        seen_hash.add(digest)
                        im.save(f"{RAW}/{cls}/bing-{q.replace(' ', '-')}-{i}.jpg", 'JPEG', quality=92)
                        added += 1
                except Exception:
                    pass                                    # Buzilgan fayl → o'tkazib yuboramiz
        print(f'{cls}: +{added} ta Bing rasmi')

for c in CLASSES:
    print(f'{c:12s} {len(os.listdir(f"{RAW}/{c}")):5d} ta rasm')

## 4. ✋ Qo'lda ko'rish (tez)
Rasmlar raqam bilan chiqadi. **Faqat aniq noto'g'rilarini** (boshqa mahsulot, mahsulot ko'rinmaydi) o'chiring.
1. `CLS` va `PAGE` ni o'zgartirib hamma sahifalarni ko'ring, yomonlarining raqamini yozib oling
2. Keyin raqamlarni pastdagi `BAD` ga yozib **bir marta** ishga tushiring (o'chirgandan keyin raqamlar siljiydi!)

In [ ]:
CLS = 'cleanser'   # cleanser, toner, serum, moisturizer, sunscreen, lip
PAGE = 0           # 0 → 0-59, 1 → 60-119, ...

names = sorted(os.listdir(f'{RAW}/{CLS}'))          # Har doim bir xil tartib
start = PAGE * 60
plt.figure(figsize=(16, 14))
for i, name in enumerate(names[start:start + 60]):
    plt.subplot(6, 10, i + 1)
    plt.imshow(Image.open(f'{RAW}/{CLS}/{name}'))
    plt.title(start + i, fontsize=10)               # Rasm raqami
    plt.axis('off')
plt.suptitle(f'{CLS}: jami {len(names)} ta | sahifa {PAGE}')
plt.tight_layout()
plt.show()

In [ ]:
BAD = []   # ← Yomon rasmlar raqamlari, masalan [3, 17, 42]. Bo'sh qolsa hech narsa o'chmaydi

names = sorted(os.listdir(f'{RAW}/{CLS}'))          # Yuqoridagi bilan bir xil tartib
for i in sorted(set(BAD), reverse=True):
    os.remove(f'{RAW}/{CLS}/{names[i]}')
print(CLS, '→ qoldi:', len(os.listdir(f'{RAW}/{CLS}')))

## 5. Train / Val / Test (70 / 15 / 15)
Bitta asl rasmdan bir nechta crop chiqishi mumkin → ular **bir guruh** bo'lib, birga bitta to'plamga tushadi (train va test aralashib ketmasin).
`random.seed(42)` → har safar bir xil bo'linadi.

In [ ]:
def group_of(name):
    """'manba__aslrasm__annid.jpg' → 'manba__aslrasm'; Bing rasmlari → o'zi bitta guruh"""
    parts = os.path.splitext(name)[0].split('__')
    return '__'.join(parts[:2]) if len(parts) >= 3 else parts[0]

random.seed(42)
if os.path.exists(DATA):
    shutil.rmtree(DATA)           # Eski bo'linishni o'chiramiz
for split in ['train', 'val', 'test']:
    for cls in CLASSES:
        os.makedirs(f'{DATA}/{split}/{cls}', exist_ok=True)

for cls in CLASSES:
    groups = defaultdict(list)
    for name in sorted(os.listdir(f'{RAW}/{cls}')):
        groups[group_of(name)].append(name)
    keys = sorted(groups)
    random.shuffle(keys)
    n = len(keys)
    n_train, n_val = int(n * 0.70), int(n * 0.15)
    parts = {'train': keys[:n_train], 'val': keys[n_train:n_train + n_val], 'test': keys[n_train + n_val:]}
    counts = {}
    for split, ks in parts.items():
        files = [f for k in ks for f in groups[k]]
        for f in files:
            shutil.copy(f'{RAW}/{cls}/{f}', f'{DATA}/{split}/{cls}/{f}')
        counts[split] = len(files)
    warn = '  ⚠️ kam — USE_BING bilan to\'ldiring' if counts['train'] < 60 else ''
    print(f"{cls:12s} train {counts['train']:4d} | val {counts['val']:3d} | test {counts['test']:3d}{warn}")